# Draft captions and VQA answers for people to verify (P3-J6)

For every **phase of a clip in one camera's view**, a vision-language model drafts a caption and answers
the questions in `config/vqa_bank.yaml`. The drafts are written as Label Studio *pre-annotations*: the
annotator sees the form already filled in, and checks or corrects it. That is much faster than writing
every caption from nothing, and the correction rate (see `annotation-kit convert`) says whether the
drafts were worth having.

**Input**: `phase_labels.jsonl` from the phase annotation kit (one clip per line) and the clip videos.
**Output**: `preannotated_tasks.json`, to import into a Label Studio project (Import, upload; the
predictions are picked up as pre-annotations).

Run it on a **Kaggle T4 (16 GB)** with Internet on. Qwen2.5-VL-7B is loaded in 4-bit.
Set `FAKE_MODEL = True` first to check the whole pipeline on any CPU in seconds; it drafts placeholder
text instead of calling a model.

The model is only ever a *draft*: an answer it gives that is not one of the allowed ones, or a reply that
is not JSON, becomes "Cannot tell" and is counted, never a confident wrong label.

In [ ]:
# 1. Get the code (Kaggle: Internet on). The kit and the question bank need only pydantic, PyYAML.
import os
import subprocess
import sys

REPO_URL = "https://github.com/SlingggShottt/genai-vms.git"
BRANCH = "main"
REPO_DIR = os.environ.get("ANNOTATION_REPO_DIR", "/kaggle/working/genai-vms")

if not os.path.exists(REPO_DIR):
    subprocess.run(  # noqa: S603 - a fixed command, no outside input
        ["git", "clone", "--depth", "1", "--branch", BRANCH, REPO_URL, REPO_DIR],  # noqa: S607
        check=True,
    )
for sub in ("libs/vms_common/src", "tools/annotation_kit/src"):
    sys.path.insert(0, f"{REPO_DIR}/{sub}")

In [ ]:
# 2. What to label, and with what.
import json
from pathlib import Path

PHASE_LABELS = os.environ.get("PHASE_LABELS", "/kaggle/input/phase-labels/phase_labels.jsonl")
VQA_BANK = f"{REPO_DIR}/config/vqa_bank.yaml"

# A clip's `video_uri` is a storage uri (s3://...); on Kaggle the same files are a dataset:
# the uri's prefix is swapped for the folder they were uploaded to.
URI_PREFIX = os.environ.get("URI_PREFIX", "s3://vms-evidence/")
VIDEO_ROOT = os.environ.get("VIDEO_ROOT", "/kaggle/input/incident-clips/")

OUT_DIR = Path(os.environ.get("OUT_DIR", "/kaggle/working"))
LIMIT = (
    int(os.environ["LIMIT"]) if os.environ.get("LIMIT") else None
)  # label only the first N tasks (a trial run)

FAKE_MODEL = (
    os.environ.get("FAKE_MODEL", "0") == "1"
)  # True: placeholder drafts, no GPU, no videos needed
MODEL_ID = "Qwen/Qwen2.5-VL-7B-Instruct"
LOAD_IN_4BIT = True
MAX_FRAMES = 8  # sampled evenly across the phase
MAX_SIDE = 640  # longest side of a frame, in pixels (VRAM: lower this if you run out)
MAX_NEW_TOKENS = 400
CHECKPOINT_EVERY = 10  # tasks; a crashed or timed-out run resumes from the file

In [ ]:
# 3. Load the question bank and expand the clips into tasks (one per phase per view).
from collections import Counter

from annotation_kit.prelabel import PROMPT_VERSION
from annotation_kit.tasks import read_phase_labels, tasks_from_clips
from vms_common.vqa_bank import load_vqa_bank

bank = load_vqa_bank(VQA_BANK)
clips = read_phase_labels(PHASE_LABELS)
tasks = list(tasks_from_clips(clips))
if LIMIT:
    tasks = tasks[:LIMIT]

MODEL_VERSION = (
    "fake-model@" + PROMPT_VERSION
    if FAKE_MODEL
    else f"{MODEL_ID.split('/')[-1].lower()}@{PROMPT_VERSION.replace('/', '-')}"
)
print(f"{len(clips)} clips -> {len(tasks)} tasks   model: {MODEL_VERSION}")
print("by event type:", dict(Counter(t.event_type for t in tasks)))
print("by phase:     ", dict(Counter(t.phase for t in tasks)))

In [ ]:
# 4. The model: a function from (task, prompt) to the model's raw text.
import re


def local_path(task):
    uri = task.video_uri
    return VIDEO_ROOT + uri[len(URI_PREFIX) :] if uri.startswith(URI_PREFIX) else uri


def fake_labeler(task, prompt):
    # Placeholder text, each question answered with its first allowed answer (a pipeline check).
    answers = {
        m.group(1): m.group(2).split(" | ")[0]
        for m in re.finditer(r"^- (\S+): .*? Answers: (.+)$", prompt, re.M)
    }
    return json.dumps(
        {
            "caption": f"Placeholder caption for the {task.phase} phase in {task.view}.",
            "answers": answers,
        }
    )


def load_qwen():
    import torch
    from qwen_vl_utils import process_vision_info
    from transformers import AutoProcessor, BitsAndBytesConfig, Qwen2_5_VLForConditionalGeneration

    quant = (
        BitsAndBytesConfig(
            load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_compute_dtype=torch.float16
        )
        if LOAD_IN_4BIT
        else None
    )
    model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
        MODEL_ID, torch_dtype=torch.float16, device_map="auto", quantization_config=quant
    )
    processor = AutoProcessor.from_pretrained(MODEL_ID)

    def frames(task):
        import cv2
        from PIL import Image

        capture = cv2.VideoCapture(local_path(task))
        if not capture.isOpened():
            raise FileNotFoundError(local_path(task))
        times = [
            task.start_s + (task.end_s - task.start_s) * (i + 0.5) / MAX_FRAMES
            for i in range(MAX_FRAMES)
        ]
        out = []
        for t in times:
            capture.set(cv2.CAP_PROP_POS_MSEC, t * 1000)
            ok, bgr = capture.read()
            if not ok:
                continue
            image = Image.fromarray(cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB))
            image.thumbnail((MAX_SIDE, MAX_SIDE))
            out.append((t, image))
        capture.release()
        if not out:
            raise ValueError(
                f"no frames read from {local_path(task)} between {task.start_s}s and {task.end_s}s"
            )
        return out

    def labeler(task, prompt):
        content = []
        for i, (t, image) in enumerate(frames(task), start=1):
            content += [
                {"type": "text", "text": f"Frame {i} @ {t:.1f} s:"},
                {"type": "image", "image": image},
            ]
        content.append({"type": "text", "text": prompt})
        messages = [{"role": "user", "content": content}]
        text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        images, videos = process_vision_info(messages)
        inputs = processor(
            text=[text], images=images, videos=videos, padding=True, return_tensors="pt"
        ).to(model.device)
        with torch.no_grad():
            generated = model.generate(**inputs, max_new_tokens=MAX_NEW_TOKENS, do_sample=False)
        new_tokens = generated[:, inputs.input_ids.shape[1] :]
        return processor.batch_decode(new_tokens, skip_special_tokens=True)[0]

    return labeler


labeler = fake_labeler if FAKE_MODEL else load_qwen()

In [ ]:
# 5. Draft, checkpointing as we go. A failure on one clip (a missing video, say) is recorded and
#    the task is left for the annotator to do from a blank; it does not stop the run.
from annotation_kit.prelabel import PrelabelStats, preannotate
from annotation_kit.tasks import dump_json

OUT_DIR.mkdir(parents=True, exist_ok=True)
OUT = OUT_DIR / "preannotated_tasks.json"


def key_of(item):
    d = item["data"]
    return f"{d['clip_id']}|{d['view']}|{d['phase']}"


items = json.loads(OUT.read_text()) if OUT.exists() else []
done = {key_of(item) for item in items}
todo = [t for t in tasks if t.key not in done]
print(f"{len(done)} already done, {len(todo)} to do")

failures = []


def safe(task, prompt):
    try:
        return labeler(task, prompt)
    except Exception as error:  # noqa: BLE001 - one bad clip must not end the run
        failures.append((task.key, repr(error)))
        return ""


stats = PrelabelStats()
for start in range(0, len(todo), CHECKPOINT_EVERY):
    batch = todo[start : start + CHECKPOINT_EVERY]
    new_items, batch_stats = preannotate(batch, bank, safe, model_version=MODEL_VERSION)
    items.extend(new_items)
    for name in ("tasks", "unparseable", "without_caption", "coerced_answers", "answers"):
        setattr(stats, name, getattr(stats, name) + getattr(batch_stats, name))
    dump_json(items, OUT)
    print(f"{len(items)}/{len(done) + len(todo)} tasks written", end="\r")

In [ ]:
# 6. How it went, and what to do next.
print(f"\n{stats.tasks} drafted this run -> {OUT}")
print(f"  replies with no usable JSON (annotator starts blank): {stats.unparseable}")
print(f"  tasks with no caption: {stats.without_caption}")
print(
    f"  answers forced to 'Cannot tell' by a problem: "
    f"{stats.coerced_answers} of {stats.answers} ({stats.coerced_rate:.1%})"
)
for key, error in failures[:10]:
    print("  FAILED", key, error)
if failures:
    print(f"  ... {len(failures)} failures in all (their tasks have no draft)")
sample = next((i for i in items if i["predictions"]), None)
if sample:
    caption = next(r for r in sample["predictions"][0]["result"] if r["from_name"] == "caption")[
        "value"
    ]["text"][0]
    print("\nSample draft:", key_of(sample), "->", caption)
print(
    "\nNext: create a Label Studio project per event type, with its config from"
    "\nml/annotation/caption_vqa/<event type>.xml; import this file (Import > upload), annotate,"
    "\nthen Export > JSON and run `annotation-kit convert`."
)